# OdorNet Baseline Training

This notebook follows the two baseline model families defined in the original reference notebook: MolFormer fine-tuning for SMILES multi-label classification and a simple two-layer GCN/GNN. Metric logging is simplified into reusable helpers while the modeling choices remain aligned with the reference implementation.

## 0. Setup

MolFormer can be loaded either from a local `molformer_config/` directory at the repository root or directly from Hugging Face with `AutoTokenizer.from_pretrained()` and `AutoModel.from_pretrained()`. The local directory is ignored by git. Download the model files from https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct and place them under `molformer_config/`. If the local folder is absent, the training function falls back to online download.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from odornet.datasets import LABEL_COLUMNS, load_odornet
from odornet.training import (
    DEFAULT_MOLFORMER_MODEL,
    MolFormerMLP,
    TrainingConfig,
    load_molformer,
    resolve_molformer_source,
    train_gnn_baseline,
    train_molformer_baseline,
)

LOCAL_MOLFORMER_PATH = ROOT / "molformer_config"
MOLFORMER_LOCAL_PATH = LOCAL_MOLFORMER_PATH if LOCAL_MOLFORMER_PATH.exists() else None
MOLFORMER_SOURCE, MOLFORMER_LOCAL_ONLY = resolve_molformer_source(
    model_name=DEFAULT_MOLFORMER_MODEL,
    local_model_path=MOLFORMER_LOCAL_PATH,
)

plt.rcParams.update({"figure.dpi": 130, "axes.grid": True})
print(f"Repository root: {ROOT}")
print(f"Default MolFormer model: {DEFAULT_MOLFORMER_MODEL}")
print(f"MolFormer source: {MOLFORMER_SOURCE}")
print(f"MolFormer local-only load: {MOLFORMER_LOCAL_ONLY}")

## 1. Load Fixed Train/Validation Split

The fixed split is the same split used by the baseline reference notebook after path cleanup: `data/processed/dataset_train_aligned.csv` and `data/processed/dataset_test_aligned.csv`. The held-out file is used as the validation/evaluation split in this repository.

In [ ]:
train_df = load_odornet("train", root=ROOT)
val_df = load_odornet("test", root=ROOT)

print("train", train_df.shape)
print("validation", val_df.shape)
print("train/validation overlap", len(set(train_df.SMILES) & set(val_df.SMILES)))

label_stats = pd.DataFrame(
    {
        "train_positive": train_df[LABEL_COLUMNS].apply(pd.to_numeric, errors="coerce").sum(),
        "train_missing": train_df[LABEL_COLUMNS].isna().sum(),
        "validation_positive": val_df[LABEL_COLUMNS].apply(pd.to_numeric, errors="coerce").sum(),
    }
).sort_values("train_positive", ascending=False)
display(label_stats)

label_stats[["train_positive", "validation_positive"]].plot(kind="bar", figsize=(11, 4), color=["#52796f", "#f4a259"])
plt.title("Positive label counts in fixed split")
plt.ylabel("Positive count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 2. Training Configuration

Defaults mirror the reference notebook: batch size 48, 30 epochs, threshold 0.5, weighted masked BCE loss, and missing-label policies `drop`, `union`, and `intersection`. For a quick smoke test, reduce `num_epochs` to 1.

In [ ]:
NAN_POLICY = "drop"  # one of: drop, union, intersection
CONFIG = TrainingConfig(
    nan_policy=NAN_POLICY,
    batch_size=48,
    num_epochs=30,
    threshold=0.5,
    seed=42,
    output_dir=str(ROOT / "outputs" / "baseline"),
)

CONFIG

## 3. MolFormer Baseline

This is the transformer-style baseline from the reference notebook: tokenize SMILES, load MolFormer, mean-pool hidden states with the attention mask, then use the MLP head `[hidden, 512, 384, 256] -> labels`. `MOLFORMER_SOURCE` points to local `molformer_config/` when available; otherwise it points to `ibm-research/MoLFormer-XL-both-10pct` and triggers Hugging Face download.

In [ ]:
molformer_result = train_molformer_baseline(
    train_df=train_df,
    test_df=val_df,
    config=CONFIG,
    model_name=DEFAULT_MOLFORMER_MODEL,
    local_model_path=MOLFORMER_LOCAL_PATH,
    labels=LABEL_COLUMNS,
)
print("Best MolFormer macro F1:", molformer_result["best_macro_f1"])


## 4. Simple GNN Baseline

This follows the reference GNN baseline: RDKit converts SMILES to molecular graphs; atom symbols are one-hot encoded; a two-layer GCN with global mean pooling predicts the 12 labels.

In [ ]:

gnn_result = train_gnn_baseline(
    train_df=train_df,
    test_df=val_df,
    config=CONFIG,
    labels=LABEL_COLUMNS,
)
print("Best GNN macro F1:", gnn_result["best_macro_f1"])


## 5. Compact Metric Review

Training writes `training_logs.json`, `best_val_macrof1.pt`, and `best_per_label_metrics.csv` under `outputs/baseline/<model>/<policy>/`. The helper below loads those files and plots validation curves.

In [ ]:
def load_training_logs(model_name: str, nan_policy: str = NAN_POLICY):
    log_path = ROOT / "outputs" / "baseline" / model_name / nan_policy / "training_logs.json"
    if not log_path.exists():
        print(f"No log file found: {log_path}")
        return None
    with log_path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    logs = pd.DataFrame(payload["logs"])
    per_label = pd.DataFrame(payload.get("best_per_label", []))
    return logs, per_label

def plot_training_logs(model_name: str, nan_policy: str = NAN_POLICY):
    loaded = load_training_logs(model_name, nan_policy)
    if loaded is None:
        return
    logs, per_label = loaded
    display(logs.tail())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    logs.plot(x="epoch", y=["train_macro_f1", "val_macro_f1", "train_micro_f1", "val_micro_f1"], ax=axes[0])
    axes[0].set_title(f"{model_name} F1 curves")
    axes[0].set_ylim(0, 1)
    logs.plot(x="epoch", y=["train_macro_auroc", "val_macro_auroc", "train_micro_auroc", "val_micro_auroc"], ax=axes[1])
    axes[1].set_title(f"{model_name} AUROC curves")
    axes[1].set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
    if not per_label.empty:
        per_label.sort_values("f1").plot(kind="barh", x="label", y="f1", figsize=(8, 5), legend=False, color="#4464ad")
        plt.title(f"{model_name} best per-label F1")
        plt.xlabel("F1")
        plt.xlim(0, 1)
        plt.tight_layout()
        plt.show()

plot_training_logs("molformer", NAN_POLICY)
plot_training_logs("gnn", NAN_POLICY)